# Matching Network Inference Testing

This notebook demonstrates how to use the `MatchingNetworkInference` class to perform few-shot image classification with a trained Matching Network model.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
from tqdm.notebook import tqdm

from matching_network import MatchingNetworkInference

## Helper Functions for Visualization

In [ ]:
def load_image(path):
    """Load an image from path for display."""
    return Image.open(path).convert('RGB')

def plot_image_grid(images, titles=None, cols=5, figsize=(15, 10), title_fontsize=8):
    """Plot a grid of images with optional titles."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i])
            if titles is not None and i < len(titles):
                ax.set_title(titles[i], fontsize=title_fontsize)
            ax.axis('off')
        else:
            ax.axis('off')
    
    plt.tight_layout()
    return fig

def display_support_set(image_paths, labels):
    """Display the support set images with their labels."""
    images = [load_image(path) for path in image_paths]
    titles = [f"Class: {label}" for label in labels]
    return plot_image_grid(images, titles, cols=5, figsize=(15, 10))

def display_query_results(image_paths, predictions):
    """Display query images with their predicted labels and confidence scores."""
    images = [load_image(path) for path in image_paths]
    titles = [f"Pred: {pred[0]}\nConf: {pred[1]:.2f}" for pred in predictions]
    return plot_image_grid(images, titles, cols=5, figsize=(15, 10))

def display_query_with_ground_truth(image_paths, predictions, true_labels):
    """Display query images with predictions and ground truth labels."""
    images = [load_image(path) for path in image_paths]
    titles = [f"True: {true}\nPred: {pred[0]} ({pred[1]:.2f})" 
              for true, pred in zip(true_labels, predictions)]
    return plot_image_grid(images, titles, cols=5, figsize=(15, 10), title_fontsize=7)

## Metrics Calculation and Visualization

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate performance metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    
    # Handle potential errors with multi-class recall and F1
    try:
        recall_micro = recall_score(y_true, y_pred, average='micro')
        recall_macro = recall_score(y_true, y_pred, average='macro')
        f1_micro = f1_score(y_true, y_pred, average='micro')
        f1_macro = f1_score(y_true, y_pred, average='macro')
    except Exception as e:
        print(f"Warning: {e}")
        recall_micro = recall_macro = f1_micro = f1_macro = float('nan')
    
    return {
        'accuracy': accuracy,
        'recall_micro': recall_micro,
        'recall_macro': recall_macro,
        'f1_micro': f1_micro,
        'f1_macro': f1_macro
    }

def plot_confusion_matrix(y_true, y_pred, class_names=None):
    """Plot a confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    return plt.gcf()

## Load the Pre-trained Model

In [ ]:
# Path to the pre-trained model
model_path = "output/best_matchingnet_model.pth"  # Update this to your actual model path

# Initialize the inference class
try:
    inference = MatchingNetworkInference(model_path)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")

## Prepare Support and Query Sets

In [ ]:
# Replace these paths with your actual dataset paths
dataset_path = "archive1"  # Update this to your actual dataset path
test_csv_path = os.path.join(dataset_path, "test.csv")

# Load test data
try:
    import pandas as pd
    test_data = pd.read_csv(test_csv_path)
    print(f"Loaded test data with {len(test_data)} images")
except Exception as e:
    print(f"Error loading test data: {e}")
    # Create dummy data if loading fails
    test_data = pd.DataFrame({
        'filename': glob.glob(os.path.join(dataset_path, "images", "*.jpg"))[:100],
        'label': ['unknown'] * 100
    })

In [ ]:
# Function to create a balanced support set with n_way classes and k_shot examples per class
def create_support_set(data, n_way=5, k_shot=5):
    # Group by label and sample n_way classes
    class_groups = data.groupby('label')
    selected_classes = np.random.choice(list(class_groups.groups.keys()), n_way, replace=False)
    
    support_images = []
    support_labels = []
    
    # Sample k_shot examples from each class
    for cls in selected_classes:
        class_samples = class_groups.get_group(cls).sample(k_shot)
        for _, row in class_samples.iterrows():
            img_path = os.path.join(dataset_path, "images", row['filename'])
            support_images.append(img_path)
            support_labels.append(cls)
    
    return support_images, support_labels, selected_classes

# Function to create a query set from remaining samples of the selected classes
def create_query_set(data, selected_classes, support_image_paths, n_query_per_class=15):
    # Extract filenames from the support images (removing path and file extension)
    support_filenames = [os.path.basename(path) for path in support_image_paths]
    
    query_images = []
    query_labels = []
    
    # Sample query examples from each class, excluding support examples
    for cls in selected_classes:
        class_samples = data[data['label'] == cls]
        # Filter out support examples
        available_samples = class_samples[~class_samples['filename'].isin(support_filenames)]
        
        # Sample n_query_per_class or all available if fewer
        num_to_sample = min(n_query_per_class, len(available_samples))
        if num_to_sample > 0:
            sampled_queries = available_samples.sample(num_to_sample)
            for _, row in sampled_queries.iterrows():
                img_path = os.path.join(dataset_path, "images", row['filename'])
                query_images.append(img_path)
                query_labels.append(cls)
    
    return query_images, query_labels

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Create support and query sets
n_way = 5  # Number of classes
k_shot = 5  # Number of support examples per class
n_query = 10  # Number of query examples per class

support_images, support_labels, selected_classes = create_support_set(test_data, n_way, k_shot)
query_images, query_labels = create_query_set(test_data, selected_classes, support_images, n_query)

print(f"Created support set with {len(support_images)} images from {n_way} classes")
print(f"Created query set with {len(query_images)} images")

# Set up the support set for the model
inference.set_support(support_images, support_labels)

# Display the support set
fig = display_support_set(support_images, support_labels)
plt.suptitle("Support Set Images", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])

## Make Predictions on Query Images

In [ ]:
# Make predictions on query images
predictions = inference.predict(query_images)

# Display query images with predictions
fig = display_query_with_ground_truth(query_images, predictions, query_labels)
plt.suptitle("Query Set Images with Predictions", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])

## Evaluate Model Performance

In [ ]:
# Extract predicted labels from predictions
y_true = query_labels
y_pred = [pred[0] for pred in predictions]  # First element of each prediction tuple is the predicted label

# Calculate metrics
metrics = calculate_metrics(y_true, y_pred)

# Print metrics
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Recall (micro): {metrics['recall_micro']:.4f}")
print(f"Recall (macro): {metrics['recall_macro']:.4f}")
print(f"F1 Score (micro): {metrics['f1_micro']:.4f}")
print(f"F1 Score (macro): {metrics['f1_macro']:.4f}")

# Display classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Plot confusion matrix
fig = plot_confusion_matrix(y_true, y_pred, class_names=selected_classes)

## Running Multiple Episodes for More Robust Evaluation

In [ ]:
def run_evaluation_episodes(data, n_episodes=10, n_way=5, k_shot=5, n_query=15):
    """Run multiple episodes and calculate average metrics."""
    episode_metrics = []
    
    for episode in tqdm(range(n_episodes), desc="Running episodes"):
        # Create support and query sets
        support_images, support_labels, selected_classes = create_support_set(data, n_way, k_shot)
        query_images, query_labels = create_query_set(data, selected_classes, support_images, n_query)
        
        # Set up support set
        inference.set_support(support_images, support_labels)
        
        # Make predictions
        predictions = inference.predict(query_images)
        
        # Extract predicted labels
        y_true = query_labels
        y_pred = [pred[0] for pred in predictions]
        
        # Calculate metrics
        metrics = calculate_metrics(y_true, y_pred)
        episode_metrics.append(metrics)
    
    # Calculate average metrics
    avg_metrics = {}
    for key in episode_metrics[0].keys():
        avg_metrics[key] = np.mean([m[key] for m in episode_metrics])
        
    return avg_metrics

In [ ]:
# Run multiple evaluation episodes
n_episodes = 5  # Change this to a higher number for more robust evaluation
avg_metrics = run_evaluation_episodes(test_data, n_episodes=n_episodes, n_way=n_way, k_shot=k_shot, n_query=n_query)

print(f"Results averaged over {n_episodes} episodes:")
print(f"Accuracy: {avg_metrics['accuracy']:.4f}")
print(f"Recall (micro): {avg_metrics['recall_micro']:.4f}")
print(f"Recall (macro): {avg_metrics['recall_macro']:.4f}")
print(f"F1 Score (micro): {avg_metrics['f1_micro']:.4f}")
print(f"F1 Score (macro): {avg_metrics['f1_macro']:.4f}")

## Exploring the Effect of Shot Count on Performance

In [ ]:
def evaluate_shot_count_effect(data, shot_counts=[1, 3, 5, 10], n_episodes=3, n_way=5, n_query=10):
    """Evaluate the effect of shot count on model performance."""
    results = []
    
    for k_shot in shot_counts:
        print(f"Evaluating with {k_shot}-shot learning...")
        avg_metrics = run_evaluation_episodes(data, n_episodes=n_episodes, 
                                              n_way=n_way, k_shot=k_shot, n_query=n_query)
        results.append({
            'k_shot': k_shot,
            'accuracy': avg_metrics['accuracy'],
            'f1_macro': avg_metrics['f1_macro']
        })
    
    return results

In [ ]:
# Evaluate the effect of shot count
shot_results = evaluate_shot_count_effect(test_data, shot_counts=[1, 3, 5], n_episodes=3)

# Plot the results
plt.figure(figsize=(10, 6))
shots = [r['k_shot'] for r in shot_results]
accuracies = [r['accuracy'] for r in shot_results]
f1_scores = [r['f1_macro'] for r in shot_results]

plt.plot(shots, accuracies, 'o-', label='Accuracy')
plt.plot(shots, f1_scores, 's-', label='F1 Score (macro)')
plt.xlabel('Number of Support Examples per Class (k-shot)')
plt.ylabel('Performance Metric')
plt.title('Effect of Support Set Size on Model Performance')
plt.xticks(shots)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

## Conclusion

In this notebook, we have demonstrated how to use the `MatchingNetworkInference` class to perform few-shot image classification. We've shown how to:

1. Set up a support set with labeled examples
2. Make predictions on new query images
3. Visualize the support and query images with labels and predictions
4. Calculate performance metrics like accuracy, recall, and F1 score
5. Analyze the effect of shot count on model performance

This approach enables quick adaptation to new tasks with minimal labeled examples, which is particularly useful in scenarios where labeled data is scarce.